# Fetch RA/Dec from VizieR

Fetches RA/Dec from Gaia DR3 via VizieR (`I/355/gaiadr3`) for stars in `training_stars.csv` that are missing coordinates (Mamonova2025 and LWRD). ESA TAP is not required.

Patches `cf_data/training_stars.csv` in place.

In [4]:
import pandas as pd
import numpy as np
import requests
from io import StringIO
from pathlib import Path
import time

CF_DATA = Path('cf_data')

VIZIER_TAP_URL = 'https://tapvizier.cds.unistra.fr/TAPVizieR/tap/sync'

def query_vizier_gaia(source_ids, chunk_size=500):
    """Query Gaia DR3 RA/Dec via CDS TAPVizieR for a list of source IDs."""
    results = []
    chunks = [source_ids[i:i+chunk_size] for i in range(0, len(source_ids), chunk_size)]
    for i, chunk in enumerate(chunks):
        if i % 5 == 0:
            print(f'  Chunk {i+1}/{len(chunks)} ({len(chunk)} IDs)')
        id_list = ','.join(str(x) for x in chunk)
        query = f'SELECT Source, RA_ICRS, DE_ICRS FROM \"I/355/gaiadr3\" WHERE Source IN ({id_list})'
        try:
            resp = requests.post(VIZIER_TAP_URL,
                data={'REQUEST':'doQuery','LANG':'ADQL','FORMAT':'csv','QUERY':query},
                timeout=60)
            if resp.status_code == 200 and resp.text.strip():
                df = pd.read_csv(StringIO(resp.text))
                if len(df) > 0:
                    results.append(df)
        except Exception as e:
            print(f'  Error on chunk {i+1}: {e}')
        time.sleep(0.5)
    if results:
        return pd.concat(results, ignore_index=True)
    return pd.DataFrame(columns=['Source', 'RA_ICRS', 'DE_ICRS'])

In [5]:
train = pd.read_csv(CF_DATA / 'training_stars.csv', dtype={'source_id': 'Int64'})
lcgen = pd.read_csv(CF_DATA / 'lcgen_rotators.csv', dtype={'GaiaDR3_ID': 'Int64'})

missing = train[train['ra'].isna() & train['source_id'].notna()].copy()
print(f'Stars missing RA/Dec: {len(missing)}')
print(missing['source_paper'].value_counts())
print()

# The source_ids for Mamonova/LWRD are float64-rounded (wrong).
# lcgen_rotators.csv has the correct GaiaDR3_IDs, matched by star name.
name_to_correct_id = lcgen.set_index('object')['GaiaDR3_ID'].to_dict()
missing['correct_source_id'] = missing['star_name'].map(name_to_correct_id)

matched = missing['correct_source_id'].notna().sum()
unmatched = (~missing['correct_source_id'].notna()).sum()
print(f'Matched to correct source_id via star name: {matched}/{len(missing)}')
print(f'Unmatched (will query with existing ID): {unmatched}')

Stars missing RA/Dec: 284
source_paper
Mamonova2025    250
LWRD             34
Name: count, dtype: int64

Matched to correct source_id via star name: 284/284
Unmatched (will query with existing ID): 0


In [6]:
## Query VizieR for RA/Dec using corrected source IDs
query_ids = missing['correct_source_id'].combine_first(missing['source_id']).dropna().astype(int).unique().tolist()
print(f'Querying {len(query_ids)} source IDs from VizieR Gaia DR3...')

coords_df = query_vizier_gaia(query_ids)
coords_df = coords_df.rename(columns={'Source': 'source_id', 'RA_ICRS': 'ra', 'DE_ICRS': 'dec'})
coords_df['source_id'] = coords_df['source_id'].astype('Int64')
print(f'Retrieved coords for {len(coords_df)} / {len(query_ids)} stars')

Querying 280 source IDs from VizieR Gaia DR3...
  Chunk 1/1 (280 IDs)


/var/folders/8q/8t0cm1_d2rldpq2v7kdzgpfw0000gn/T/ipykernel_69122/2349713918.py:2: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  query_ids = missing['correct_source_id'].combine_first(missing['source_id']).dropna().astype(int).unique().tolist()


Retrieved coords for 280 / 280 stars


In [7]:
train = pd.read_csv(CF_DATA / 'training_stars.csv', dtype={'source_id': 'Int64'})
lcgen = pd.read_csv(CF_DATA / 'lcgen_rotators.csv', dtype={'GaiaDR3_ID': 'Int64'})

# Fix rounded source_ids for Mamonova/LWRD using lcgen name lookup
name_to_correct_id = lcgen.set_index('object')['GaiaDR3_ID'].to_dict()
rounded_mask = train['source_paper'].isin(['Mamonova2025', 'LWRD'])
train.loc[rounded_mask, 'source_id'] = train.loc[rounded_mask, 'star_name'].map(name_to_correct_id).combine_first(
    train.loc[rounded_mask, 'source_id']
)

# Merge RA/Dec — coords_df is keyed on correct source_ids
train = train.merge(coords_df[['source_id', 'ra', 'dec']], on='source_id', how='left', suffixes=('', '_new'))
train['ra'] = train['ra'].combine_first(train['ra_new'])
train['dec'] = train['dec'].combine_first(train['dec_new'])
train = train.drop(columns=['ra_new', 'dec_new'])

print(f'ra missing after patch: {train["ra"].isna().sum()}/{len(train)}')
print(train.groupby('source_paper')['ra'].apply(lambda x: f'{x.notna().sum()}/{len(x)} have ra').to_string())

train.to_csv(CF_DATA / 'training_stars.csv', index=False)
print('\nSaved training_stars.csv')

ra missing after patch: 0/4923
source_paper
Engle2023           41/41 have ra
LWRD                38/38 have ra
MOCADB          4503/4503 have ra
Mamonova2025      331/331 have ra
Pass2022            10/10 have ra

Saved training_stars.csv


/var/folders/8q/8t0cm1_d2rldpq2v7kdzgpfw0000gn/T/ipykernel_69122/3952201711.py:7: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  train.loc[rounded_mask, 'source_id'] = train.loc[rounded_mask, 'star_name'].map(name_to_correct_id).combine_first(


## Patch training_stars.csv

## Fetch missing parallax_error from VizieR

In [10]:
# Patch parallax_error into all 4 CSVs
id_to_plx_err = plx_err_df.set_index('source_id')['parallax_error'].to_dict()

for f, col in [
    (CF_DATA / 'training_stars.csv', 'source_id'),
    (CF_DATA / 'validation_stars.csv', 'source_id'),
    (CF_DATA / 'mmcoeval.csv', 'source_id'),
]:
    df = pd.read_csv(f, dtype={col: 'Int64'})
    mask = df['parallax_error'].isna() & df[col].notna()
    df.loc[mask, 'parallax_error'] = df.loc[mask, col].map(id_to_plx_err)
    print(f'{f.name}: parallax_error missing after patch: {df["parallax_error"].isna().sum()}/{len(df)}')
    df.to_csv(f, index=False)

# cfrc
df = pd.read_csv(CF_DATA / 'cf_rotator_catalog.csv', dtype={'GaiaDR3_ID': 'Int64'})
mask = df['parallax_error'].isna() & df['GaiaDR3_ID'].notna()
df.loc[mask, 'parallax_error'] = df.loc[mask, 'GaiaDR3_ID'].map(id_to_plx_err)
print(f'cf_rotator_catalog.csv: parallax_error missing after patch: {df["parallax_error"].isna().sum()}/{len(df)}')
df.to_csv(CF_DATA / 'cf_rotator_catalog.csv', index=False)

print('\nDone.')

training_stars.csv: parallax_error missing after patch: 34/4923
validation_stars.csv: parallax_error missing after patch: 7/242
mmcoeval.csv: parallax_error missing after patch: 0/34
cf_rotator_catalog.csv: parallax_error missing after patch: 0/5832

Done.


In [9]:
# Fetch parallax_error from VizieR for all stars missing it across all 4 catalogs
train = pd.read_csv(CF_DATA / 'training_stars.csv', dtype={'source_id': 'Int64'})
val   = pd.read_csv(CF_DATA / 'validation_stars.csv', dtype={'source_id': 'Int64'})
mm    = pd.read_csv(CF_DATA / 'mmcoeval.csv', dtype={'source_id': 'Int64'})
cfrc  = pd.read_csv(CF_DATA / 'cf_rotator_catalog.csv', dtype={'GaiaDR3_ID': 'Int64'})

# Collect all source_ids missing parallax_error
missing_err = []
for df, col in [(train,'source_id'),(val,'source_id'),(mm,'source_id')]:
    m = df[df['parallax_error'].isna() & df[col].notna()]
    missing_err.extend(m[col].astype(int).tolist())
m = cfrc[cfrc['parallax_error'].isna() & cfrc['GaiaDR3_ID'].notna()]
missing_err.extend(m['GaiaDR3_ID'].astype(int).tolist())
missing_err = list(set(missing_err))
print(f'Stars missing parallax_error: {len(missing_err)}')

# Query VizieR
def query_vizier_gaia_plx(source_ids, chunk_size=500):
    results = []
    chunks = [source_ids[i:i+chunk_size] for i in range(0, len(source_ids), chunk_size)]
    for i, chunk in enumerate(chunks):
        if i % 5 == 0:
            print(f'  Chunk {i+1}/{len(chunks)}')
        id_list = ','.join(str(x) for x in chunk)
        query = f'SELECT Source, e_Plx FROM \"I/355/gaiadr3\" WHERE Source IN ({id_list})'
        try:
            resp = requests.post(VIZIER_TAP_URL,
                data={'REQUEST':'doQuery','LANG':'ADQL','FORMAT':'csv','QUERY':query},
                timeout=60)
            if resp.status_code == 200 and resp.text.strip():
                df = pd.read_csv(StringIO(resp.text))
                if len(df) > 0:
                    results.append(df)
        except Exception as e:
            print(f'  Error: {e}')
        time.sleep(0.5)
    if results:
        return pd.concat(results, ignore_index=True)
    return pd.DataFrame(columns=['Source', 'e_Plx'])

plx_err_df = query_vizier_gaia_plx(missing_err)
plx_err_df = plx_err_df.rename(columns={'Source': 'source_id', 'e_Plx': 'parallax_error'})
plx_err_df['source_id'] = plx_err_df['source_id'].astype('Int64')
print(f'Retrieved parallax_error for {len(plx_err_df)} / {len(missing_err)} stars')

Stars missing parallax_error: 4877
  Chunk 1/10
  Chunk 6/10
Retrieved parallax_error for 4872 / 4877 stars


In [ ]:
# Patch recovered parallax/parallax_error back into training_stars.csv and validation_stars.csv
train = pd.read_csv(CF_DATA / 'training_stars.csv', dtype={'source_id': 'Int64'})
val   = pd.read_csv(CF_DATA / 'validation_stars.csv', dtype={'source_id': 'Int64'})

print(f'Recovered {len(recovered)} stars via SIMBAD/cone search')

for df, name in [(train, 'training_stars'), (val, 'validation_stars')]:
    patched = 0
    for old_sid, info in recovered.items():
        mask = df['source_id'] == old_sid
        if mask.any():
            df.loc[mask, 'parallax']       = info['parallax']
            df.loc[mask, 'parallax_error'] = info['parallax_error']
            # Update source_id if SIMBAD found a different DR3 ID
            if info['new_source_id'] != old_sid:
                df.loc[mask, 'source_id'] = info['new_source_id']
            patched += 1
    print(f'{name}: patched {patched} rows')
    print(f'  parallax missing after: {df["parallax"].isna().sum()}/{len(df)}')

train.to_csv(CF_DATA / 'training_stars.csv', index=False)
val.to_csv(CF_DATA / 'validation_stars.csv', index=False)
print('\nSaved.')

In [8]:
train = pd.read_csv(CF_DATA / 'training_stars.csv', dtype={'source_id': 'Int64'})

train = train.merge(coords_df[['source_id', 'ra', 'dec']], on='source_id', how='left', suffixes=('', '_new'))
train['ra'] = train['ra'].combine_first(train['ra_new'])
train['dec'] = train['dec'].combine_first(train['dec_new'])
train = train.drop(columns=['ra_new', 'dec_new'])

print(f'ra missing after patch: {train["ra"].isna().sum()}/{len(train)}')
print(train.groupby('source_paper')['ra'].apply(lambda x: f'{x.notna().sum()}/{len(x)} have ra').to_string())

train.to_csv(CF_DATA / 'training_stars.csv', index=False)
print('\nSaved training_stars.csv')

ra missing after patch: 0/4923
source_paper
Engle2023           41/41 have ra
LWRD                38/38 have ra
MOCADB          4503/4503 have ra
Mamonova2025      331/331 have ra
Pass2022            10/10 have ra

Saved training_stars.csv
